In [ ]:
import pandas as pd
import ast
from collections import Counter
from itertools import combinations
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv('datasets/majority_voting_results.csv')

In [ ]:
def count_spans(merged_entities):
    if isinstance(merged_entities, str):
        count = 0
        spans = ast.literal_eval(merged_entities)
        for span in spans:
            if span['aspects']:
                count += 1
        return count
    return 0

In [ ]:
df['num_spans'] = df['merged_entities'].apply(count_spans)

print("Average number of spans in each sample: ", df['num_spans'].mean())
print("Maximum number of spans: ", df['num_spans'].max())
print("Minimum number of spans: ", df['num_spans'].min())

In [ ]:
# Plot counts of spans per sample as integer bar chart and save as PDF
# Aggregate counts per integer span value
counts = df['num_spans'].value_counts().sort_index()
min_spans = int(df['num_spans'].min())
max_spans = int(df['num_spans'].max())
x = list(range(min_spans, max_spans + 1))
counts = counts.reindex(x, fill_value=0)

fig, ax = plt.subplots(figsize=(12, 6))
palette = sns.color_palette("Blues", n_colors=len(x))
bars = ax.bar(x, counts.values, color=palette, edgecolor='black', linewidth=0.6)

# Annotate counts above each bar
max_count = counts.values.max() if len(counts) else 0
offset = max(1, int(max_count * 0.02))
for xi, v in zip(x, counts.values):
    ax.text(xi, v + offset, f'{int(v)}', ha='center', va='bottom', fontsize=11)

ax.set_xlabel('Number of Spans per Sample', fontsize=14)
ax.set_ylabel('Number of Samples', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(x, fontsize=12)
ax.tick_params(axis='y', labelsize=12)

# Clean up plot appearance
sns.despine(trim=False)
plt.tight_layout()

# Save as PDF for paper
output_path = 'outputs/spans_per_sample.pdf'
fig.savefig(output_path, format='pdf', dpi=300, bbox_inches='tight')
print('Saved spans-per-sample plot to', output_path)
fig

In [ ]:
span_lengths = []
for row in df['merged_entities']:
    for span in ast.literal_eval(row):
        span_lengths.append(span['end'] - span['start'])

print("Average length of spans: ", sum(span_lengths) / len(span_lengths))

In [ ]:
# Span length distribution — publication-quality histogram and save as PDF
import matplotlib as mpl
mpl.rcParams.update({'font.size': 12, 'axes.titlesize': 14, 'axes.labelsize': 14})
import seaborn as sns
sns.set_style('white')

fig, ax = plt.subplots(figsize=(10, 5))

# Histogram
bins = 25
counts, bins_edges, patches = ax.hist(span_lengths, bins=bins, edgecolor='black', color=sns.color_palette('Blues', n_colors=1)[0])

# Annotate counts above each bin
max_count = counts.max() if len(counts) else 0
offset = max(1, int(max_count * 0.02))
for c, left, right in zip(counts, bins_edges[:-1], bins_edges[1:]):
    if c > 0:
        ax.text((left + right) / 2, c + offset, f'{int(c)}', ha='center', va='bottom', fontsize=10)

ax.set_ylabel('Number of Spans', fontsize=14)
ax.set_xlabel('Span Length in Characters', fontsize=13)

# Turn off grid lines
ax.grid(False)

sns.despine(trim=False)
plt.tight_layout()

# Save as PDF for paper
output_path = 'outputs/span_length_distribution.pdf'
fig.savefig(output_path, format='pdf', dpi=300, bbox_inches='tight')
print('Saved span length plot to', output_path)
fig

In [ ]:
span_records = []

for row in df['merged_entities']:
    for span in ast.literal_eval(row):
        for aspect in span['aspects']:
            span_records.append({
                'aspect': aspect,
                'length': span['end'] - span['start']
            })

span_df = pd.DataFrame(span_records)

In [ ]:
# Boxplot of span length per aspect — publication-quality and save as PDF
import matplotlib as mpl
mpl.rcParams.update({'font.size': 12, 'axes.titlesize': 14, 'axes.labelsize': 14})
import seaborn as sns
sns.set_style('white')

fig, ax = plt.subplots(figsize=(10, 6))

# Draw boxplot using seaborn on the provided axes
sns.boxplot(data=span_df, x='aspect', y='length', color='lightblue', ax=ax)

# Labels and ticks
ax.set_ylabel('Span Length in Characters', fontsize=14)
ax.set_xlabel('Aspects', fontsize=13)
plt.setp(ax.get_xticklabels(), rotation=30, fontsize=11)

# Turn off grid lines for publication look
ax.grid(False)

sns.despine(trim=False)
plt.tight_layout()

# Save as PDF
output_path = 'outputs/span_length_by_aspect.pdf'
fig.savefig(output_path, format='pdf', dpi=300, bbox_inches='tight')
print('Saved boxplot to', output_path)
fig

In [ ]:
cooc_counter = Counter()

for row in df['merged_entities']:
    entities = row if isinstance(row, list) else ast.literal_eval(row)
    aspects = set()
    for entity in entities:
        aspects.update(entity.get('aspects', []))
    
    if len(aspects) > 1:
        for pair in combinations(sorted(aspects), 2):
            cooc_counter[pair] += 1

In [ ]:
# Co-occurrence heatmap — publication-quality and save as PDF
import matplotlib as mpl
mpl.rcParams.update({'font.size': 12, 'axes.titlesize': 14, 'axes.labelsize': 14})
import seaborn as sns
sns.set_style('white')

aspects = sorted({asp for pair in cooc_counter for asp in pair})
matrix = pd.DataFrame(0, index=aspects, columns=aspects)
for (a1, a2), count in cooc_counter.items():
    matrix.loc[a1, a2] = count
    matrix.loc[a2, a1] = count

fig, ax = plt.subplots(figsize=(12, 10))
# Use integer annotation, centered, and a perceptually uniform colormap
sns.heatmap(matrix, annot=True, fmt='d', cmap='Blues', cbar_kws={'label': 'Co-occurrence Count'}, ax=ax)

# Tweak appearance for publication
ax.set_xlabel('Aspects', fontsize=13)
ax.set_ylabel('Aspects', fontsize=13)
plt.setp(ax.get_xticklabels(), rotation=45, ha='right', fontsize=10)
plt.setp(ax.get_yticklabels(), rotation=0, fontsize=10)

# Remove gridlines/background clutter
ax.grid(False)
sns.despine(left=True, bottom=True)
plt.tight_layout()

# Save as PDF
output_path = 'outputs/cooccurrence_heatmap.pdf'
fig.savefig(output_path, format='pdf', dpi=300, bbox_inches='tight')
print('Saved co-occurrence heatmap to', output_path)
fig

In [ ]:

# ── Dataset Statistics Table ──────────────────────────────────────────────────
from collections import Counter
import ast

aspect_counts_per_sample = []
sentiment_pos = sentiment_neu = sentiment_neg = 0
aspect_counter = Counter()
spans_per_sample = []
span_word_lengths = []

for idx, row in df.iterrows():
    entities = row['merged_entities']
    if not isinstance(entities, str):
        continue
    spans = ast.literal_eval(entities)
    text = row['text'] if isinstance(row['text'], str) else ''

    # aspects per sample (count all aspect occurrences, no dedup)
    sample_aspect_count = 0
    sample_span_count = 0
    for span in spans:
        aspects = span.get('aspects', [])
        sentiments = span.get('sentiments', [])
        if aspects:
            sample_span_count += 1
            sample_aspect_count += len(aspects)
            aspect_counter.update(aspects)
        # aspect-sentiment pairs
        for s in sentiments:
            if s == 'Positive':
                sentiment_pos += 1
            elif s == 'Neutral':
                sentiment_neu += 1
            elif s == 'Negative':
                sentiment_neg += 1
        # span word length
        span_text = text[span['start']:span['end']]
        span_word_lengths.append(len(span_text.split()))

    aspect_counts_per_sample.append(sample_aspect_count)
    spans_per_sample.append(sample_span_count)

import numpy as np

most_freq_aspect  = aspect_counter.most_common(1)[0][0]
least_freq_aspect = aspect_counter.most_common()[-1][0]
multi_aspect_pct  = (sum(1 for c in aspect_counts_per_sample if c > 1) / len(aspect_counts_per_sample)) * 100

stats = {
    'Avg. # Aspects per Sample':           f'{np.mean(aspect_counts_per_sample):.3f}',
    'Min # Aspects per Sample':            int(np.min(aspect_counts_per_sample)),
    'Max # Aspects per Sample':            int(np.max(aspect_counts_per_sample)),
    'Samples with Multiple Aspects (%)':   f'{multi_aspect_pct:.1f}%',
    'Most Frequent Aspect':                most_freq_aspect,
    'Least Frequent Aspect':               least_freq_aspect,
    'Positive Aspect-Sentiment Pairs':     sentiment_pos,
    'Neutral Aspect-Sentiment Pairs':      sentiment_neu,
    'Negative Aspect-Sentiment Pairs':     sentiment_neg,
    'Avg. # Spans per Sample':             f'{np.mean(spans_per_sample):.3f}',
    'Min # Spans per Sample':              int(np.min(spans_per_sample)),
    'Max # Spans per Sample':              int(np.max(spans_per_sample)),
    'Avg. Evidence Span Length (words)':   f'{np.mean(span_word_lengths):.3f}',
    'Min Evidence Span Length (words)':    int(np.min(span_word_lengths)),
    'Max Evidence Span Length (words)':    int(np.max(span_word_lengths)),
}

stats_df = pd.DataFrame(list(stats.items()), columns=['Statistic', 'Value'])
print(stats_df.to_string(index=False))
